# First hitting times

## Load data

In [ ]:
import json
from math import ceil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from lib import (
    N_OPTIMIZERS,
    N_PROBLEMS,
    OPTIMIZER_OVERVIEW_PATH,
    PROBLEM_OVERVIEW_PATH,
    RELATIVE_WALLTIME_LIMIT,
)

optimizer_df = pd.read_csv(OPTIMIZER_OVERVIEW_PATH)
problem_df = pd.read_csv(PROBLEM_OVERVIEW_PATH)

with open("../data/first_hitting_times_marvin.json") as f:
    data = json.load(f)

df_fht_all = pd.DataFrame(data)

df_fht_all = df_fht_all.join(
    problem_df.set_index(["short"]),
    on="problem",
    how="left",
    validate="many_to_one",
)
df_fht_all = df_fht_all.join(
    optimizer_df.set_index(["output_dir"]),
    on="optimizer",
    how="left",
    validate="many_to_one",
)

# separate rows per significance level
df_fht_all = df_fht_all.explode(
    ["times", "thresholds", "significance_levels"]
).rename(
    columns={
        "significance_levels": "significance_level",
        "thresholds": "fval_threshold",
        "times": "first_hitting_time_s",
    }
)
df_fht_all.first_hitting_time_s = df_fht_all.first_hitting_time_s.astype(float)
df_fht_all = df_fht_all.query("excluded == False")
assert df_fht_all.first_hitting_time_s.notna().all()
df_fht_no_pyscat = df_fht_all.query("~is_pysacess")
df_fht_all

In [ ]:
# Check data completeness
for alpha in df_fht_no_pyscat.significance_level.unique():
    complete = (
        df_fht_no_pyscat.query(f"significance_level == {alpha}")
        .groupby(["problem", "optimizer"])
        .size()
        .unstack(fill_value=0)
    )

    assert (complete == 1).all().all(), display(complete)
    # Note: Keep all first_hitting_time_s==inf rows.
    #  They are needed to compute proper ranks further down
    assert complete.shape == (N_PROBLEMS, N_OPTIMIZERS)

In [ ]:
df_fht_all.query("optimizer == 'BFGS' and problem=='Beer'")

## First-hitting times optimizer x problem (Fig 5B)

In [ ]:
def plot_fht(
    ax1: plt.Axes | None = None,
    ax2: plt.Axes | None = None,
    alpha=0.05,
    legend_ncol=2,
):
    if ax1 is None:
        fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2)

    df = df_fht_no_pyscat.query(f"significance_level == {alpha}")
    assert len(df) == N_PROBLEMS * N_OPTIMIZERS
    # order by median
    order = (
        df.replace(np.inf, np.nan)
        .groupby(["problem", "difficulty"])
        .agg({"first_hitting_time_s": "median"})
        .sort_values("first_hitting_time_s")
        .reset_index()
        .set_index("problem")
    )
    # insert placeholder between easy and hard
    order = (
        order.query("difficulty == 'easy'").index.tolist()
        + [""]
        + order.query("difficulty != 'easy'").index.tolist()
    )

    # barplot of number of successful optimizers
    ax = ax2
    df["solved"] = np.isfinite(df.first_hitting_time_s)
    num_solved_df = (
        df.groupby("problem").solved.sum().rename("num_solved").to_frame()
    )
    num_solved_df = num_solved_df.reindex(order)
    sns.barplot(
        data=num_solved_df,
        y=num_solved_df.index,
        x="num_solved",
        ax=ax,
        color="lightgray",
        edgecolor="black",
        legend=False,
    )
    ax.set_ylabel("Problem")
    ax.set_xlabel("# Successful\noptimisation methods")
    ax1.sharey(ax2)
    ax.yaxis.set_visible(False)

    # FHT
    ax = ax1
    cmap = plt.get_cmap("gray")
    colors = [cmap(v) for v in np.linspace(0.05, 0.95, len(order))]
    colors = dict(zip(order, colors, strict=False))
    optimizer_colors = (
        optimizer_df.set_index("optimizer_label")
        .sort_index()
        .optimizer_color.to_dict()
    )
    sns.boxplot(
        data=df,
        order=order,
        x="first_hitting_time_s",
        y="problem",
        hue="problem",
        palette=colors,
        showfliers=False,
        ax=ax,
        log_scale=True,
    )
    sns.stripplot(
        data=df,
        order=order,
        x="first_hitting_time_s",
        y="problem",
        hue="optimizer_label",
        hue_order=sorted(df.optimizer_label, key=str.lower),
        jitter=0.2,
        ax=ax,
        alpha=0.8,
        size=3,
        palette=optimizer_colors,
    )
    ax.legend(bbox_to_anchor=(1.01, 1.01), ncol=legend_ncol)
    ax.set_xlabel("First hitting time (s)")
    ax.set_ylabel("Problem")

    # hide tick at placeholder x
    gap_idx = order.index("")
    tick = ax.yaxis.get_major_ticks()[gap_idx]
    tick.tick1line.set_visible(False)
    tick.tick2line.set_visible(False)

    # lines for walltime limit
    ax.vlines(
        x=RELATIVE_WALLTIME_LIMIT * 60 * 60 * 3,
        ymin=0,
        ymax=gap_idx,
        # colors="darkgreen",
        colors="black",
        lw=1.5,
        ls=":",
        zorder=1,
    ).set_capstyle("round")
    ax.vlines(
        x=RELATIVE_WALLTIME_LIMIT * 60 * 60 * 9,
        ymin=gap_idx,
        ymax=len(order),
        # ymin=0, ymax=len(order),
        colors="black",
        lw=1.5,
        ls=":",
        zorder=1,
    ).set_capstyle("round")
    ax.text(
        RELATIVE_WALLTIME_LIMIT * 60 * 60 * 3,
        -0.2,
        "time limit",
        # color="darkgreen",
        ha="center",
        rotation=0,
    )

    # dotted guide lines
    y = list(range(0, len(order)))
    y.remove(gap_idx)
    ax.hlines(
        y=y,
        xmin=ax.get_xlim()[0],
        xmax=ax.get_xlim()[1],
        colors="lightgray",
        lw=1,
        ls=":",
        zorder=0,
    )
    ax.set_ylim(bottom=len(order), top=-1.2)

    ax.get_legend().set_bbox_to_anchor((2.3, 1))

In [ ]:
# Compute ranking
alpha = 0.05
df_rank = (
    df_fht_no_pyscat.query(f"significance_level == {alpha}")
    .set_index(["problem", "optimizer_label"])
    .groupby("problem")
    .first_hitting_time_s.rank()
    .rename("rank")
    .to_frame()
)

assert df_rank["rank"].notna().all()

df_rank.query("rank == 1")

In [ ]:
# how often is each optimizer the fastest?
def plot_freq_rank1(df_rank: pd.DataFrame, ax: plt.Axes = None):
    import matplotlib.colors as mcolors

    df_rank1 = (
        df_rank.query("rank == 1")
        .reset_index()["optimizer_label"]
        .value_counts()
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))

    optimizer_colors = (
        optimizer_df.set_index("optimizer_label")
        .sort_index()
        .optimizer_color.to_dict()
    )
    colors = [optimizer_colors[optimizer] for optimizer in df_rank1.index]
    colors = [mcolors.to_rgba(c, alpha=0.6) for c in colors]

    ax.bar(df_rank1.index, df_rank1.values, color=colors, edgecolor="black")
    ax.tick_params(axis="x", labelrotation=90)
    # ax.set_ylabel("# problems with rank 1")
    ax.set_ylabel("# Problems solved fastest")
    ax.set_xlim(left=-0.6, right=len(df_rank1) - 0.4)

    # some are not solved at all
    assert df_rank1.sum() <= N_PROBLEMS

In [ ]:
def plot_median_rank(df_rank: pd.DataFrame, ax: plt.Axes | None = None):
    import matplotlib.colors as mcolors

    df_median_rank = (
        df_rank.groupby("optimizer_label")
        .agg({"rank": "median"})
        .rename(columns={"rank": "median_rank"})
        .sort_values("median_rank")
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 4))

    optimizer_colors = (
        optimizer_df.set_index("optimizer_label")
        .sort_index()
        .optimizer_color.to_dict()
    )
    colors = [
        optimizer_colors[optimizer] for optimizer in df_median_rank.index
    ]
    colors = [mcolors.to_rgba(c, alpha=0.6) for c in colors]

    ax.bar(
        df_median_rank.index,
        df_median_rank.median_rank,
        color=colors,
        edgecolor="black",
    )
    ax.tick_params(axis="x", labelrotation=90)
    ax.set_ylabel("Median rank")
    ax.set_xlim(left=-0.6, right=len(df_median_rank) - 0.4)
    ax.set_ylim(bottom=0, top=5 * ceil(df_median_rank.median_rank.max() / 5))

In [ ]:
## Figure 5:

with plt.rc_context(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial"],
        "font.size": 7,
        "axes.labelsize": 7,
        "axes.titlesize": 7,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 5,
        "savefig.bbox": None,
        "lines.markersize": 1,
        "lines.linewidth": 0.8,
        "patch.linewidth": 0.8,
    }
):
    fig = plt.figure(
        figsize=(6.5, 10),
        dpi=300,
    )  # layout="constrained")
    gs = fig.add_gridspec(
        3,
        3,
        width_ratios=[3, 0.5, 1],
        height_ratios=[1, 5, 2],
        hspace=1.1,
        left=0.05,
        right=0.95,
        top=0.9,
    )

    ax_top = fig.add_subplot(gs[0, :])
    ax_center_left = fig.add_subplot(gs[1, 0])
    ax_center_right = fig.add_subplot(gs[1, 1])
    ax_bottom_left = fig.add_subplot(gs[2, 0])
    ax_bottom_right = fig.add_subplot(gs[2, 2])
    subfig_font = {
        "fontsize": 9,
        "fontweight": "bold",
    }

    ax = ax_top
    ax_top.remove()

    fig.text(0.0, 0.7, "B", va="top", ha="left", **subfig_font)
    fig.text(
        0.05,
        0.7,
        "First hitting times of optimisation methods",
        va="top",
        ha="left",
        **subfig_font,
    )
    plot_fht(ax1=ax_center_left, ax2=ax_center_right, legend_ncol=1)
    ax_center_left.get_legend().set_bbox_to_anchor((1.7, 1))

    fig.text(0.0, 0.3, "C", va="top", ha="left", **subfig_font)
    fig.text(
        0.05,
        0.3,
        "Median ranking of optimisation methods",
        va="top",
        ha="left",
        **subfig_font,
    )

    fig.text(0.75, 0.3, "D", va="top", ha="left", **subfig_font)
    fig.text(
        0.8,
        0.3,
        "Fastest optimisation methods",
        va="top",
        ha="left",
        **subfig_font,
    )

    plot_median_rank(df_rank, ax=ax_bottom_left)
    plot_freq_rank1(df_rank, ax=ax_bottom_right)

    plt.savefig("out/Figure5.svg", bbox_inches="tight")
    plt.savefig("out/Figure5.pdf", bbox_inches="tight")
    plt.show()